In [3]:
# ============================================================
# Data Exploration
# ============================================================

import json
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [29]:
import os

# Find where you are right now
print("Current directory:", os.getcwd())

# List everything in current folder
print("\nFiles here:")
for item in os.listdir('.'):
    print(item)

Current directory: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/notebooks

Files here:
01_data_exploration.ipynb
02_workflow_1_direct_generation.ipynb
.ipynb_checkpoints


In [30]:
# Search for CSV files anywhere in your project
for root, dirs, files in os.walk('.'):
    # Skip venv folder
    if 'venv' in root:
        continue
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

In [31]:
import pandas as pd

# Update these paths to match where your files are
notes = pd.read_csv('../data/raw/clinical_notes.csv')
patients = pd.read_csv('../data/raw/patients.csv')
admissions = pd.read_csv('../data/raw/admissions.csv')

In [32]:
# Basic shape
print("=== NOTES ===")
print(notes.shape)
print(notes.columns.tolist())
print(notes.head(2))

print("\n=== PATIENTS ===")
print(patients.shape)
print(patients.columns.tolist())

print("\n=== ADMISSIONS ===")
print(admissions.shape)
print(admissions.columns.tolist())

=== NOTES ===
(1602, 9)
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
   ingest_timestamp                      clinical_note_id  \
0  07/01/2026 14:35  17bf845b-88f8-4604-8983-6e74453aada5   
1  07/01/2026 14:50  e5d9c0a4-299a-425e-abbc-27fabe9cb742   

                                     clean_note_text creation_timestamp  \
0  Patient Name: Judith Ada Wells\n- Patient ID: ...   07/01/2026 14:05   
1  Patient reviewed at 14:20 on 07/01/26 by Nurse...   07/01/2026 14:20   

         updt_dt_tm         note_subject note_type  \
0  07/01/2026 14:35            ED Triage        ED   
1  07/01/2026 14:50  ED Triage Follow-Up        ED   

                           admission_id                             person_id  
0  63720303-3c1b-4356-befd-eea5438da62e  28570119-9cdc-4120-98c0-4edb76cf36a3  
1  63720303-3c1b-4356-befd-eea5438da62e  28570119-9cdc-4120-98c0-4edb76cf36a3  

=== PATI

In [33]:
print("\n=== NOTE TYPES ===")
print(notes['note_type'].value_counts())


=== NOTE TYPES ===
note_type
Orthopaedics Inpatients                      263
Physiotherapy Documentation                  262
Medicine Inpatients                          242
ED                                           155
Neurology Inpatients                         104
Therapies Inpatients                          81
Paediatrics Inpatients                        66
Anaesthetic Documentation                     62
Respiratory Inpatients                        62
ED Depart Summary                             39
Respiratory Medicine Inpatients               37
Pre-op Checklist                              35
Pre-op Consent                                31
Theatre notes                                 31
Neurosurgery Inpatients                       26
Orthopaedics Outpatients                      24
Occupational Therapy Documentation            24
Surgery Inpatients                            21
Dietetics Documentation                       14
Speech and Language Therapy Documentati

In [34]:
# How many notes per admission?
notes_per_admission = notes.groupby('admission_id')['clinical_note_id'].count()
print("\n=== NOTES PER ADMISSION ===")
print(notes_per_admission.describe())
print(notes_per_admission.value_counts().head(10))

# How many admissions per patient?
admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
print("\n=== ADMISSIONS PER PATIENT ===")
print(admissions_per_patient.describe())
print(admissions_per_patient.value_counts().head(10))


=== NOTES PER ADMISSION ===
count    69.000000
mean     23.217391
std       8.709139
min      10.000000
25%      16.000000
50%      22.000000
75%      30.000000
max      45.000000
Name: clinical_note_id, dtype: float64
clinical_note_id
22    5
25    5
33    4
17    4
23    4
19    4
35    4
15    4
12    3
26    3
Name: count, dtype: int64

=== ADMISSIONS PER PATIENT ===
count    50.000000
mean      1.380000
std       0.490314
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       2.000000
Name: admission_id, dtype: float64
admission_id
1    31
2    19
Name: count, dtype: int64


In [35]:
# Pick first patient and show all their notes in order
first_patient = notes['person_id'].iloc[0]
patient_notes = notes[notes['person_id'] == first_patient].sort_values('creation_timestamp')

print(f"\n=== FULL JOURNEY FOR PATIENT {first_patient} ===")
for _, row in patient_notes.iterrows():
    print(f"\nDate: {row['creation_timestamp']}")
    print(f"Type: {row['note_type']}")
    print(f"Text preview: {str(row['clean_note_text'])[:200]}")
    print("---")


=== FULL JOURNEY FOR PATIENT 28570119-9cdc-4120-98c0-4edb76cf36a3 ===

Date: 07/01/2026 14:05
Type: ED
Text preview: Patient Name: Judith Ada Wells
- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3
- NHS Number: 272733208
- Date of Birth: 15/05/84 (39 years old)
- Gender: Female
- Allergies: NKA
- Current Medicatio
---

Date: 07/01/2026 14:20
Type: ED
Text preview: Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurologic
---

Date: 07/01/2026 14:45
Type: ED
Text preview: - Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.
 - Date/Time: 07/01/26, 14:45.
 - Staff involved: Nurse Jasmine Freda Murray.
 - Chief Complaint: Severe headach
---

Date: 07/01/2026 15:15
Type: ED
Text preview: Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current 

In [36]:
# Check duration of admissions
notes['creation_timestamp'] = pd.to_datetime(
    notes['creation_timestamp'], 
    format='%d/%m/%Y %H:%M',
    dayfirst=True
)

date_range = notes.groupby('admission_id')['creation_timestamp'].agg(['min', 'max'])
date_range['duration_days'] = (date_range['max'] - date_range['min']).dt.days

print("=== ADMISSION DURATION ===")
print(date_range['duration_days'].describe())

# Check patients with 2 admissions
admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
two_admission_patients = admissions_per_patient[admissions_per_patient == 2].index.tolist()
print(f"\nPatients with 2 admissions: {len(two_admission_patients)}")

# Check note types for multi-admission patients
multi_notes = notes[notes['person_id'].isin(two_admission_patients)]
print(f"Total notes for multi-admission patients: {len(multi_notes)}")

# Find discharge summary note type
print("\n=== DISCHARGE SUMMARY NOTES ===")
discharge_notes = notes[notes['note_type'].str.contains('Depart|Discharge|Summary', 
                                                          case=False, na=False)]
print(f"Count: {len(discharge_notes)}")
print(discharge_notes['note_type'].value_counts())

=== ADMISSION DURATION ===
count    69.000000
mean      8.869565
std       6.216470
min       0.000000
25%       4.000000
50%       7.000000
75%      15.000000
max      20.000000
Name: duration_days, dtype: float64

Patients with 2 admissions: 19
Total notes for multi-admission patients: 988

=== DISCHARGE SUMMARY NOTES ===
Count: 39
note_type
ED Depart Summary    39
Name: count, dtype: int64


In [37]:
# First confirm column names in both tables
print("Notes columns:", notes.columns.tolist())
print("Admissions columns:", admissions.columns.tolist())

# Check if person_id and patient_id contain same values
print("\nSample person_id from notes:", notes['person_id'].iloc[0])
print("Sample patient_id from admissions:", admissions['patient_id'].iloc[0])

Notes columns: ['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
Admissions columns: ['admission_id', 'admission_method', 'admission_timestamp', 'admission_title', 'bed_location', 'date_of_birth', 'nhs_number', 'patient_id', 'patient_name', 'site_id', 'site_name', 'ward', 'full_name', 'first_name', 'surname']

Sample person_id from notes: 28570119-9cdc-4120-98c0-4edb76cf36a3
Sample patient_id from admissions: 28570119-9cdc-4120-98c0-4edb76cf36a3


In [38]:
import pandas as pd

admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
two_admission_patients = admissions_per_patient[admissions_per_patient >= 2].index.tolist()
print(f"Patients with 2 admissions: {len(two_admission_patients)}")

Patients with 2 admissions: 19


In [39]:
multi_admission_with_gt = notes[
    (notes['person_id'].isin(two_admission_patients)) & 
    (notes['note_type'] == 'ED Depart Summary')
].groupby('person_id')['admission_id'].nunique()

both_admissions_have_gt = multi_admission_with_gt[
    multi_admission_with_gt == 2
].index.tolist()

print(f"Patients with ground truth for both admissions: {len(both_admissions_have_gt)}")

Patients with ground truth for both admissions: 10


In [40]:
# How many admissions do these 10 patients have total?
print(admissions[admissions['patient_id'].isin(both_admissions_have_gt)].groupby('patient_id')['admission_id'].count())

patient_id
04df53ea-55c1-48d9-84a1-1f15c133b29b    2
137b8481-4f1d-4b7f-babd-20f7117023ad    2
359014a1-10e6-4bd8-9ba7-513d021c971e    2
5e434d78-b2f6-4d88-b327-fff6ee50b901    2
61699d6d-904a-4ece-9026-cd61b3fd9a50    2
69bf7e25-abb2-4dde-857f-f1138d4d0d8a    2
8448b3fa-1f5e-45b5-bda5-eb0b7921b8cc    2
c50e236f-6b3d-41c8-9e16-7ec343cac820    2
c6c45c39-cd73-49dd-818d-0a7865fe8a7f    2
ff8c4724-b7de-4189-bccf-cffddd4d6d44    2
Name: admission_id, dtype: int64


In [41]:
import json

with open('../data/patients/longitudinal_patient_ids.json', 'w') as f:
    json.dump(both_admissions_have_gt, f)

print(f"Saved {len(both_admissions_have_gt)} patient IDs")

Saved 10 patient IDs


In [42]:
# Basic counts from the notes table
print("Number of notes:", len(notes))
print("Number of patients:", notes["person_id"].nunique())
print("Number of admissions:", notes["admission_id"].nunique())

# Check whether each patient has one or multiple admissions
admissions_per_patient = (
    notes.groupby("person_id")["admission_id"]
    .nunique()
)

print("\nAdmissions per patient:")
print(admissions_per_patient.describe())

print("\nPatients with more than one admission:")
print((admissions_per_patient > 1).sum())


# Calculate approximate note length in words
notes["word_count"] = (
    notes["clean_note_text"]
    .fillna("")
    .str.split()
    .str.len()
)

print("\nNote length statistics:")
print(
    notes["word_count"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

Number of notes: 1602
Number of patients: 50
Number of admissions: 69

Admissions per patient:
count    50.000000
mean      1.380000
std       0.490314
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       2.000000
Name: admission_id, dtype: float64

Patients with more than one admission:
19

Note length statistics:
count    1602.000000
mean      129.997503
std        83.061291
min         1.000000
25%        60.250000
50%       124.500000
75%       174.000000
90%       228.900000
95%       280.000000
max       561.000000
Name: word_count, dtype: float64


In [43]:
notes_per_admission = (
    notes.groupby("admission_id")
    .size()
)

print("\nNotes per admission:")
print(
    notes_per_admission.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)


Notes per admission:
count    69.000000
mean     23.217391
std       8.709139
min      10.000000
25%      16.000000
50%      22.000000
75%      30.000000
90%      35.000000
95%      36.000000
max      45.000000
dtype: float64


In [44]:
longest_notes = notes.nlargest(10, "word_count")

for _, row in longest_notes.iterrows():
    print("=" * 80)
    print("Words:", row["word_count"])
    print("Subject:", row["note_subject"])
    print("Type:", row["note_type"])
    print()
    print(row["clean_note_text"])
    print()

Words: 561
Subject: Medical Clerking
Type: Neurology Inpatients

Clerking Doctor
Dr. Kelly Nicola Hayward (SpR)

Presenting Complaint
Acute confusion following minor fall

History of Presenting Complaint
- Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26)
- No LOC reported but recalls feeling dazed afterward
- Developed pprogressive confusion over the following hours
- Unable to remember recent events clearly
- Denies headache, visual changes, N&V, or limb weakness
- No reported CP, palpitations, or SOB

Review of Systems
- CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures
- CVS: Denies chest pain, palpitations, or syncope
- Resp: No breathlessness or cough
- GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit
- Renal: No dysuria or haematuria
- MSK: Reports mild right knee pain following the fall
- Endo: No polyuria, polydipsia, or heat/cold intolerance
- Other: No r

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

In [5]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))
notes = pd.read_csv(DATA_DIR / "raw" / "clinical_notes.csv")
print("Original notes:", len(notes))
print(notes.columns.tolist())
print("Notes:", notes.shape)

Original notes: 1602
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
Notes: (1602, 9)


In [8]:
# ============================================================
# SELECT PILOT PATIENTS BY LONGITUDINAL RECORD SIZE
#
# Restrict selection to the longitudinal cohort:
# patients with >= 2 admissions and ground truth for both.
# ============================================================

# ============================================================
# RECORD SIZE PER PATIENT
# ============================================================

patient_sizes = (
    notes
    .groupby("person_id")
    .size()
    .reset_index(name="note_count")
    .sort_values("note_count")
    .reset_index(drop=True)
)

display(patient_sizes)

print("\nNote-count distribution:")
print(patient_sizes["note_count"].describe())

,person_id,note_count
0,c87e610e-ac2a-48cf-ac32-054f3e595498,10
1,58b8aad6-7327-4450-956f-b775be0f4984,11
2,96766f3f-aa50-4bcb-abbf-884328049ea9,11
3,a8629d2e-7fd8-449a-9149-e0f5045a04c4,12
4,f97ee974-67f3-4d72-9ed4-f17025da3749,13
5,c54a9649-495a-4f89-a8ca-fbb14ba70ca6,14
6,900efb9a-fb57-4f25-b2a3-0a8c6f55b2d6,15
7,028998ee-babc-4096-9b28-001bc2f9a84e,15
8,0f438665-d430-4adb-8acc-c3beed9e4942,15
9,f1538231-21af-4cfa-8b40-3303aa0d0c96,15



Note-count distribution:
count    50.000000
mean     32.040000
std      19.973411
min      10.000000
25%      17.000000
50%      24.500000
75%      45.500000
max      90.000000
Name: note_count, dtype: float64


In [9]:
notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Raw notes:", len(notes))
print("After cleaning + deduplication:", len(notes_dedup))

Raw notes: 1602
After cleaning + deduplication: 1103


In [10]:
patient_sizes_dedup = (
    notes_dedup
    .groupby("person_id")
    .size()
    .reset_index(name="note_count")
    .sort_values("note_count")
    .reset_index(drop=True)
)

display(patient_sizes_dedup)

print("\nDeduplicated note-count distribution:")
print(
    patient_sizes_dedup["note_count"].describe()
)

,person_id,note_count
0,ef943dfb-1368-44e0-a946-a0be360e1d54,10
1,c87e610e-ac2a-48cf-ac32-054f3e595498,10
2,96766f3f-aa50-4bcb-abbf-884328049ea9,11
3,58b8aad6-7327-4450-956f-b775be0f4984,11
4,f44d4a08-4c76-4000-ad61-d1d81032e643,12
5,a8629d2e-7fd8-449a-9149-e0f5045a04c4,12
6,f97ee974-67f3-4d72-9ed4-f17025da3749,13
7,1705dd0f-011a-492c-b006-b27e03f2f4ed,14
8,c54a9649-495a-4f89-a8ca-fbb14ba70ca6,14
9,028998ee-babc-4096-9b28-001bc2f9a84e,15



Deduplicated note-count distribution:
count    50.000000
mean     22.060000
std       8.471826
min      10.000000
25%      15.250000
50%      20.500000
75%      27.000000
max      45.000000
Name: note_count, dtype: float64


In [11]:
pilot_patients = {
    "short": "c87e610e-ac2a-48cf-ac32-054f3e595498",
    "medium": "ce0046dc-0ad3-4710-8147-549793c58b44",
    "long": "c6c45c39-cd73-49dd-818d-0a7865fe8a7f",
}

for case, patient_id in pilot_patients.items():
    n = len(
        notes_dedup[
            notes_dedup["person_id"] == patient_id
        ]
    )
    print(f"{case}: {patient_id} | {n} notes")

short: c87e610e-ac2a-48cf-ac32-054f3e595498 | 10 notes
medium: ce0046dc-0ad3-4710-8147-549793c58b44 | 21 notes
long: c6c45c39-cd73-49dd-818d-0a7865fe8a7f | 45 notes


In [12]:
# Short: 10 notes — c87e610e-ac2a-48cf-ac32-054f3e595498
# Medium: 21 notes — ce0046dc-0ad3-4710-8147-549793c58b44
# Long: 45 notes — c6c45c39-cd73-49dd-818d-0a7865fe8a7f

In [4]:
# ============================================================
# LOAD + PREPROCESS SOURCE NOTES
# ============================================================

notes = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print("Raw notes:", len(notes))
print("After cleaning + deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

Raw notes: 1602
After cleaning + deduplication: 1103
Patients: 50


In [5]:
# Patient to inspect for the clinician-evaluation pilot
PILOT_PATIENT_ID = "028998ee-babc-4096-9b28-001bc2f9a84e"

# Get this patient's complete cleaned + deduplicated clinical record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

print("Number of source notes:", len(pilot_notes))

# Display each note chronologically, exactly as it exists after
# the frozen preprocessing used in the study.
for i, row in pilot_notes.iterrows():
    print("=" * 100)
    print("Timestamp:", row["creation_timestamp"])
    print("=" * 100)
    print(row["clean_note_text"])
    print()

Number of source notes: 15
Timestamp: 07/01/2026 08:45
 - Patient: Tomos Ellis, 15-year-old male, presented via A&E on 07/01/26 at 08:45 with severe abdominal pain over the past 3 days. - Date of birth: 2008-05-17. - NHS number: 965833270. - Triage category: 3 - Urgent. - No known allergies reported. - Current medications: None declared.- Past medical history: None documented. - Initial assessment performed by Nurse Jamie Leigh Alexander. - ED diagnosis: Constipation. - Admiting consultant: Dr. Susan Jennifer Robson. - Decision: Proceed with baseline obs and assess pain severity.
Nurse Jamie Leigh Alexander 
NMC number: 27H5222T

Timestamp: 07/01/2026 09:10
Patient: Tomos Ellis, 15-year-old male, presenting with abdominal pain.

- Event date/time: 07/01/26 at 09:10.
- Performed focused abdominal assessment by Nurse Jamie Leigh Alexander.
- Findings: Mild abdo distension, tenderness in lower abdo.

- NO (No guarding, rebound tenderness).
- Advise to NPO.
- Plan: Initiate IV fluids to ma

In [6]:
import json
from pathlib import Path

# Patient we're using to design the clinician-evaluation format
PILOT_PATIENT_ID = "028998ee-babc-4096-9b28-001bc2f9a84e"

PROJECT_ROOT = Path.cwd().parent.parent
RESULTS_DIR = PROJECT_ROOT / "data" / "results"

# Frozen final result files
summary_files = {
    "direct": RESULTS_DIR / "direct" / "direct_summaries.json",
    "hierarchical": RESULTS_DIR / "hierarchical" / "hierarchical_summaries.json",
    "rag": RESULTS_DIR / "rag" / "final_rag_summaries.json",
    "rag_verification": RESULTS_DIR / "rag_verification" / "final_verified_summaries.json",
}

# Field containing the final summary in each workflow
summary_fields = {
    "direct": "summary",
    "hierarchical": "final_summary",
    "rag": "summary",
    "rag_verification": "final_summary",
}

# Load and print the exact frozen summary from each workflow
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    patient_result = results[PILOT_PATIENT_ID]
    summary = patient_result[summary_fields[workflow]]

    pilot_summaries[workflow] = summary

    print("\n" + "=" * 100)
    print(workflow.upper())
    print("=" * 100)
    print(summary)


DIRECT
Tomos Ellis, a 15-year-old male with no documented past medical history, medications, or allergies, presented via A&E on 07/01/2026 at 08:45 with 3 days of worsening constant lower abdominal pain and constipation. He reported reduced appetite and mild nausea without vomiting or fever. Examination showed mild abdominal distension and lower-quadrant tenderness without guarding, rebound tenderness, or peritonism. Mild dehydration was noted, with slightly dry mucous membranes and reduced skin turgor; observations were stable.

He was initially diagnosed with constipation. He was kept NPO/NBM and commenced on 0.9% sodium chloride IV fluids, 500 mL over 4 hours. Blood tests, including FBC, U&Es, CRP, and LFTs, were normal. Abdominal X-ray demonstrated significant faecal loading without obstruction or perforation. A single 10 mg oral dose of bisacodyl was given, and he was admitted to the paediatric ward for supportive management and observation.

On medical review, abdominal pain had

In [7]:
# ---------------------------------------------------------
# Clinician validation — Patient 01 document content
# ---------------------------------------------------------

PATIENT_LABEL = "Patient 01"

# These are presentation labels only.
# Workflow names will NOT appear in the clinician document.
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

# Build clinician-facing text
lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append("")

# Add the original cleaned/deduplicated notes chronologically
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Add the four blinded summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

patient_01_content = "\n".join(lines)

print(patient_01_content)

Patient 01

SOURCE CLINICAL RECORD

Clinical Note 1
Date/Time: 07/01/2026 08:45

- Patient: Tomos Ellis, 15-year-old male, presented via A&E on 07/01/26 at 08:45 with severe abdominal pain over the past 3 days. - Date of birth: 2008-05-17. - NHS number: 965833270. - Triage category: 3 - Urgent. - No known allergies reported. - Current medications: None declared.- Past medical history: None documented. - Initial assessment performed by Nurse Jamie Leigh Alexander. - ED diagnosis: Constipation. - Admiting consultant: Dr. Susan Jennifer Robson. - Decision: Proceed with baseline obs and assess pain severity.
Nurse Jamie Leigh Alexander 
NMC number: 27H5222T

--------------------------------------------------------------------------------

Clinical Note 2
Date/Time: 07/01/2026 09:10

Patient: Tomos Ellis, 15-year-old male, presenting with abdominal pain.

- Event date/time: 07/01/26 at 09:10.
- Performed focused abdominal assessment by Nurse Jamie Leigh Alexander.
- Findings: Mild abdo dist

In [8]:
# ---------------------------------------------------------
# Clinician validation — source record sizes
# ---------------------------------------------------------

# Calculate source-record size using ONLY the original
# cleaned/deduplicated notes — no evaluation scores.
patient_sizes = (
    notes_dedup
    .groupby("person_id")
    .agg(
        note_count=("clean_note_text", "size"),
        word_count=(
            "clean_note_text",
            lambda x: sum(len(str(note).split()) for note in x)
        )
    )
    .reset_index()
    .sort_values("word_count")
    .reset_index(drop=True)
)

print("Patients:", len(patient_sizes))
print()
print(patient_sizes[["person_id", "note_count", "word_count"]].to_string(index=False))

Patients: 50

                           person_id  note_count  word_count
c87e610e-ac2a-48cf-ac32-054f3e595498          10        1380
96766f3f-aa50-4bcb-abbf-884328049ea9          11        1508
973ee9d2-7c23-4e1d-afb2-86f679b55964          17        1770
ef943dfb-1368-44e0-a946-a0be360e1d54          10        1772
c54a9649-495a-4f89-a8ca-fbb14ba70ca6          14        1780
68258b8d-7d36-4d32-9093-6170a32bf087          18        1797
f97ee974-67f3-4d72-9ed4-f17025da3749          13        1800
29ea304f-821d-474e-81a1-394ca3945e02          17        1813
fc6268ab-ad15-4613-9d8d-12c045e2fd46          16        1840
028998ee-babc-4096-9b28-001bc2f9a84e          15        1903
a8629d2e-7fd8-449a-9149-e0f5045a04c4          12        1927
c51243b8-e3d9-4cf7-ab29-0315048f91dc          17        2024
f44d4a08-4c76-4000-ad61-d1d81032e643          12        2048
f1538231-21af-4cfa-8b40-3303aa0d0c96          15        2075
0f438665-d430-4adb-8acc-c3beed9e4942          15        2200
1705dd0f-0

In [9]:
# ---------------------------------------------------------
# Clinician validation — Patient 02
# ---------------------------------------------------------

PILOT_PATIENT_ID = "05192757-942f-460d-b4ff-004ec39cc5ee"

# Get complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 02")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 02
Source notes: 23
Source words: 2958
direct: 582 summary words
hierarchical: 484 summary words
rag: 477 summary words
rag_verification: 458 summary words


In [10]:
# ---------------------------------------------------------
# Clinician validation — Patient 02 document content
# ---------------------------------------------------------

PATIENT_LABEL = "Patient 02"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append("")

# Exact source notes — no editing or typo correction
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

patient_02_content = "\n".join(lines)

print(patient_02_content)

Patient 02

SOURCE CLINICAL RECORD

Clinical Note 1
Date/Time: 02/01/2026 07:50

Clerking Doctor
Dr. Marcus John Whitehead (SpR)

Presenting Complaint
Admitged for elective L THR due to end-stage OA and history of HTN.

Procedure
Total left hip replacement

On Examination
Pt alert, NAD. L hip: deformity noted, no erythema or warmth, tender on palpation. Limited ROM. Unable to fully bear weight on L side. LL neuro: intact. No sensory deficits. Motor function normal. Skin over surgical site intact, no signs of infxn.

Observations
HR 75
BP 142/88
RR 16
Temp 36.8
SpO2 98%

Investigations
Pre-op ECG showed normal siuns rhythm with no ischemic changes. BP noted to be mildly elevated (142/88 mmHg), consistent with the patient's history of HTN. Imaging reports not mentioned in the provided information.

Test Results
Mild anaemia (Hb 11.3 g/dL). Other routine bloods WNL.

Plan
1. Proceed with planned L THR as sched. 2. Maintain fasting pre-op. 3. Ensure anaesthetic team notified. 4. Post-op an

In [11]:
# ---------------------------------------------------------
# Clinician validation — Patient 03
# ---------------------------------------------------------

PILOT_PATIENT_ID = "ebb1ca4c-f42f-41d8-adbc-1b08949036d1"

# Get complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 03")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 03
Source notes: 22
Source words: 3436
direct: 418 summary words
hierarchical: 512 summary words
rag: 326 summary words
rag_verification: 326 summary words


In [14]:
# ---------------------------------------------------------
# Clinician validation — Patient 03 document content
# ---------------------------------------------------------

PATIENT_LABEL = "Patient 03"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_03_content = "\n".join(lines)

print(patient_03_content)

Patient 03

SOURCE CLINICAL RECORD

Clinical Note 1
Date/Time: 06/01/2026 09:05

Clerking Doctor
Dr. Laurence Walter Green (Consultant)

Presenting Complaint
Elective admission for correction of left distal femoral growth plate abnormality. History of recurrent knee pain and gait abnormality.

Procedure
Hemiepiphysiodesi s of the left distal femur

On Examination
- Alert, cooperative, and well-appearing 8-year-old female. No signs of distress.
- Left knee: No erythema, swelling, or effusion. No signs of infection at the surgical site. Skin intact with no abnormal findings. No tenderness on palpation. Full passive ROM noted.
- General physical examination: CVS, resp, and abdo systems unremarkable. No pallor, jaundice, or lymphadenopathy.
- Fasting status: Cofirmed fasting as per pre-op protocol.
- No allergies or current meds as per pre-op checklist.

Observations
HR 88
BP 102/68
RR 18
Temp 36.7
SpO2 99%

Investigations
No imaging or ECG performed on admission. Pre-op imaging reviewed p

In [15]:
# ---------------------------------------------------------
# Clinician validation — Patient 04
# ---------------------------------------------------------

PILOT_PATIENT_ID = "6e93f9d9-213d-4f2c-a1f0-f475dacef554"

# Complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 04")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 04
Source notes: 34
Source words: 3867
direct: 453 summary words
hierarchical: 496 summary words
rag: 467 summary words
rag_verification: 467 summary words


In [16]:
# ---------------------------------------------------------
# Clinician validation — Patient 03 document content
# ---------------------------------------------------------

PILOT_PATIENT_ID = "6e93f9d9-213d-4f2c-a1f0-f475dacef554"
PATIENT_LABEL = "Patient 04"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_03_content = "\n".join(lines)

print(patient_03_content)

Patient 04

SOURCE CLINICAL RECORD

Clinical Note 1
Date/Time: 02/01/2026 09:00

Pre-clerking Doctor
Dr. Donna Hayley Harvey (SpR)

Presenting Complaint
Scoliosis requiring corrective surgery.

Procedure
Posterior spinal fusion for scoliosis.

History of Presenting Complaint
Pt initially noticed uneven shoulders ~2 yrs ago. Gradual worsening w/ noticeable curvature of spine. No pain reported. Denies weakness, numbness, or bowel/bladder sx.

Past Medical History
Juvenile idiopathic scoliosis

Medications
Ferrous sulfate 200 mg OD (started today to optimise Hb levels due to borderline Hb of 11.5 g/dL).

Allergies
- Aspirin

Social History
- Lives w/ parents.
- Fully independent in ADLs.
- Non-smoker. No alcohol. No recreational drug use.
- Active, paticipates in school sports.

Family History
- No FHx of scoliosis or connective tissue disorders.
- No significant FHx of other conditions.

On Examination
Alert, aell-appearing 13 y/o F (Katie Catherine Kaur). - Gen: NAD, well-developed, no 

In [17]:
# ---------------------------------------------------------
# Clinician validation — Patient 05
# ---------------------------------------------------------

PILOT_PATIENT_ID = "137b8481-4f1d-4b7f-babd-20f7117023ad"

# Complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 05")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 05
Source notes: 36
Source words: 3991
direct: 385 summary words
hierarchical: 415 summary words
rag: 351 summary words
rag_verification: 351 summary words


In [19]:
# ---------------------------------------------------------
# Clinician validation — Patient 05 document content
# ---------------------------------------------------------

PILOT_PATIENT_ID = "137b8481-4f1d-4b7f-babd-20f7117023ad"

PATIENT_LABEL = "Patient 05"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append(f"Total clinical notes: {len(pilot_notes)}")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_05_content = "\n".join(lines)

print(patient_05_content)

Patient 05

SOURCE CLINICAL RECORD
Total clinical notes: 36

Clinical Note 1
Date/Time: 06/01/2026 11:25

Patient assessed on 06/01/26 at 11:25 by Nurse Darren Christopher Burns. Initial vital signs recorded: BP 165/92 mmHg, HR 88 bpm. Mild disorientation noted during questioning. Categorised as Cat 2 for urgent ttiage by ED team.

Chief complaint: Acute confusion after minor fall.

Past medical history: HTN, Osteoarthritis, TIA (2018).

Allergies: Pollen.

ABCDE assessment:
A: Airway patent, no stridor or obstruction.
B: Breathing unlaboured, RR within normal range, SpO2 not recorded.
C: BP165/92 mmHg, HR 88 bpm, no active haemorrhage noted.
D: Mild disorientation, GCS not formally documented.
E: No obvious external injuries noted, mild bruising on left hip reported by patient post-fall.

Directed to ED team for further urgent asssssment and management.
Nurse Darren Christopher Burns 
NMC number: 19P6622C

-------------------------------------------------------------------------------

In [20]:
# ---------------------------------------------------------
# Clinician validation — Patient 06
# ---------------------------------------------------------

PILOT_PATIENT_ID = "9d8fd25d-c4fe-4343-8ff5-5da5dad4e740"

# Complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 06")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 06
Source notes: 26
Source words: 4287
direct: 584 summary words
hierarchical: 597 summary words
rag: 414 summary words
rag_verification: 408 summary words


In [21]:
# ---------------------------------------------------------
# Clinician validation — Patient 06 document content
# ---------------------------------------------------------

PILOT_PATIENT_ID = "9d8fd25d-c4fe-4343-8ff5-5da5dad4e740"

PATIENT_LABEL = "Patient 06"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append(f"Total clinical notes: {len(pilot_notes)}")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_06_content = "\n".join(lines)

print(patient_06_content)

Patient 06

SOURCE CLINICAL RECORD
Total clinical notes: 26

Clinical Note 1
Date/Time: 05/01/2026 12:35

Clerking Doctor
Dr. Promise Nkosi (Consultant)

Presenting Complaint
Elective admission for total right shoulddr replacement.

Procedure
Total right shoulder replacement for end stage OA.

On Examination
The patient appears well and in no distress. General physical examination showed no pallor, cyanosis, jaundice, or oedema. CVS examination revealed normal heart sounds with no murmurs or added sounds. Resp examination showed clear breath sounds bilaterally with no wheeze or crackles. Focused MSK examination of the right shoulder revealed marked restriction in range of motion withcrepitus on movement. No joint instability was noted, and no erythema or swelling was present. The patienrt has a documented allergy to nuts.

Observations
HR 72
BP 138/85
RR 14
Temp 36.8
SpO2 98%

Investigations
No new imaging or ECG performed. Pre-op ECG from 23/12/25 showed minor age-related changes with

In [22]:
# ---------------------------------------------------------
# Clinician validation — Patient 07
# ---------------------------------------------------------

PILOT_PATIENT_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

# Complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 07")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 07
Source notes: 45
Source words: 5134
direct: 448 summary words
hierarchical: 468 summary words
rag: 336 summary words
rag_verification: 317 summary words


In [23]:
# ---------------------------------------------------------
# Clinician validation — Patient 07 document content
# ---------------------------------------------------------

PILOT_PATIENT_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

PATIENT_LABEL = "Patient 07"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append(f"Total clinical notes: {len(pilot_notes)}")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_07_content = "\n".join(lines)

print(patient_07_content)

Patient 07

SOURCE CLINICAL RECORD
Total clinical notes: 45

Clinical Note 1
Date/Time: 03/01/2026 01:35

Patient Abena Bonsu, a 43-year-old female, was assessed on arrival at 01:35 on 03/01/26 by Nurse Audrey Phyllis Gill. Chief complaint: Progressive breathlessness. Triage category: Category 2 - Higy risk, potentially life-threatening condition requiring urgent attention. ED diagnosis: Chronic thromboembolic pulmonary HTN. Vital signs recorded: Oxygen saturation 89% on room air, indicating mild hypoxia. Past medical history includes HTN and asthma. Allergy to nuts noted. No current medications reported by the patient. Patient placed on low-flow oxygen therapy to improve oxygenation as an immediate intervention. Admission to Respiratory High Dependency Unit (HDU) under care of Dr. Elizabeth Kathryn Singh (Consultant) planned for further management of chronic thromboembolic pulmonary HTN.
Nurse Audrey Phyllis Gill 
NMC number: 75Q7413G

-------------------------------------------------

In [25]:
# ---------------------------------------------------------
# Check Patient 08 availability across all workflows
# ---------------------------------------------------------

PILOT_PATIENT_ID = "69bf7e25-abb2-4dde-857f-f1138d4d0d8a7f"

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    print(
        workflow,
        "FOUND" if PILOT_PATIENT_ID in results else "MISSING"
    )

direct MISSING
hierarchical MISSING
rag MISSING
rag_verification MISSING


In [26]:
# ---------------------------------------------------------
# Find eligible clinician-validation patients
# ---------------------------------------------------------

# Load authoritative study cohort from the frozen Direct results
with open(summary_files["direct"], "r", encoding="utf-8") as f:
    direct_results = json.load(f)

study_ids = set(direct_results.keys())

# Keep only patients that actually belong to the frozen study cohort
eligible_patient_sizes = (
    patient_sizes[
        patient_sizes["person_id"].astype(str).isin(study_ids)
    ]
    .sort_values("word_count")
    .reset_index(drop=True)
)

print("Eligible patients:", len(eligible_patient_sizes))
print(
    eligible_patient_sizes[
        ["person_id", "note_count", "word_count"]
    ].to_string(index=False)
)

Eligible patients: 50
                           person_id  note_count  word_count
c87e610e-ac2a-48cf-ac32-054f3e595498          10        1380
96766f3f-aa50-4bcb-abbf-884328049ea9          11        1508
973ee9d2-7c23-4e1d-afb2-86f679b55964          17        1770
ef943dfb-1368-44e0-a946-a0be360e1d54          10        1772
c54a9649-495a-4f89-a8ca-fbb14ba70ca6          14        1780
68258b8d-7d36-4d32-9093-6170a32bf087          18        1797
f97ee974-67f3-4d72-9ed4-f17025da3749          13        1800
29ea304f-821d-474e-81a1-394ca3945e02          17        1813
fc6268ab-ad15-4613-9d8d-12c045e2fd46          16        1840
028998ee-babc-4096-9b28-001bc2f9a84e          15        1903
a8629d2e-7fd8-449a-9149-e0f5045a04c4          12        1927
c51243b8-e3d9-4cf7-ab29-0315048f91dc          17        2024
f44d4a08-4c76-4000-ad61-d1d81032e643          12        2048
f1538231-21af-4cfa-8b40-3303aa0d0c96          15        2075
0f438665-d430-4adb-8acc-c3beed9e4942          15        2200
17

In [27]:
# ---------------------------------------------------------
# Clinician validation — Patient 08
# ---------------------------------------------------------

PILOT_PATIENT_ID = "69bf7e25-abb2-4dde-857f-f1138d4d0d8a"

# Complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 08")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 08
Source notes: 35
Source words: 5525
direct: 482 summary words
hierarchical: 531 summary words
rag: 490 summary words
rag_verification: 488 summary words


In [28]:
# ---------------------------------------------------------
# Clinician validation — Patient 08 document content
# ---------------------------------------------------------

PILOT_PATIENT_ID = "69bf7e25-abb2-4dde-857f-f1138d4d0d8a"

PATIENT_LABEL = "Patient 08"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append(f"Total clinical notes: {len(pilot_notes)}")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_08_content = "\n".join(lines)

print(patient_08_content)

Patient 08

SOURCE CLINICAL RECORD
Total clinical notes: 35

Clinical Note 1
Date/Time: 05/01/2026 00:05

Patient assessed upon arrival at 00:05 on 2026-01-05 by Nurse Kelly Nicola Hayward. Patient is a 52-year-old male presenting with acute confusion after a minor fall. Vital signs were recorded and within normal limits.

Patient Information:
- Name: Haruto Watanabe
- Age: 52
- Gender: Male
- Allergies: Pollen
- Past Medical History: HTN, HLD
- Current Medications: None

ABCDE assessment:
A: Airway patent, no strido or obstruction.
B: Breathing normal, RR 16, SpO2 98% on room air.
C: Circulation stable, BP 128/78 mmHg, HR 72 bpm, capillary refill <2 sec.
D: GCS 14 (confusion noted), PERL, no focal deficits.
E: No signs of external injury or bruising; afebrile (37.0Â°C).

Additional Information:
- Triage Category: Category 2
- Admitting Consultant: Dr. Sadia Rahman

Next  steps: Patient prioritised for further triage and assessment due to acute confusion following recent fall.
Nurse Ke

In [29]:
# ---------------------------------------------------------
# Clinician validation — Patient 09
# ---------------------------------------------------------

PILOT_PATIENT_ID = "136c7916-4f9b-4e5c-bf01-77e9d2c681a2"

# Complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 09")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 09
Source notes: 37
Source words: 4278
direct: 404 summary words
hierarchical: 580 summary words
rag: 406 summary words
rag_verification: 394 summary words


In [30]:
# ---------------------------------------------------------
# Clinician validation — Patient 09 document content
# ---------------------------------------------------------

PILOT_PATIENT_ID = "136c7916-4f9b-4e5c-bf01-77e9d2c681a2"

PATIENT_LABEL = "Patient 09"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append(f"Total clinical notes: {len(pilot_notes)}")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_09_content = "\n".join(lines)

print(patient_09_content)

Patient 09

SOURCE CLINICAL RECORD
Total clinical notes: 37

Clinical Note 1
Date/Time: 02/01/2026 21:15

Patient Philip Paul Cook, a 49-year-old male, arrived via A&E at 21: 15 on 02/01/26 with a chief complaint of acute confusion after a minor fall. Assessed by Nurse Dawn Yvonne Gibbs. GCS scored at 14/15, with mild disorientation noted. Past medical history includes HTN and mild osteoarthritis. No allergies reported. Triage category assigned as Category 2 - Emergency (requires rapid assessment and treatment within 10 minutes). Decision made to proceed with initialtriage and neurological observation.
Nurse Dawn Yvonne Gibbs 
NMC number: 28R0973Z

--------------------------------------------------------------------------------

Clinical Note 2
Date/Time: 02/01/2026 21:30

Performed a briefneuro assessment at 21:30. Paftient remains disoriented with GCS score of 14/15, which was initially recorded upon arrival in the ED. No other focal neuro deficits noted. Assessment conducted by Nurs

In [31]:
# ---------------------------------------------------------
# Clinician validation — Patient 10
# ---------------------------------------------------------

PILOT_PATIENT_ID = "ef943dfb-1368-44e0-a946-a0be360e1d54"

# Complete cleaned/deduplicated source record
pilot_notes = (
    notes_dedup[
        notes_dedup["person_id"].astype(str) == PILOT_PATIENT_ID
    ]
    .sort_values("creation_timestamp")
    .copy()
)

# Load the four exact frozen summaries
pilot_summaries = {}

for workflow, path in summary_files.items():
    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    pilot_summaries[workflow] = (
        results[PILOT_PATIENT_ID][summary_fields[workflow]]
    )

print("Patient 10")
print("Source notes:", len(pilot_notes))
print(
    "Source words:",
    sum(len(str(x).split()) for x in pilot_notes["clean_note_text"])
)

for workflow, summary in pilot_summaries.items():
    print(f"{workflow}: {len(summary.split())} summary words")

Patient 10
Source notes: 10
Source words: 1772
direct: 471 summary words
hierarchical: 415 summary words
rag: 503 summary words
rag_verification: 490 summary words


In [32]:
# ---------------------------------------------------------
# Clinician validation — Patient 10 document content
# ---------------------------------------------------------

PILOT_PATIENT_ID = "ef943dfb-1368-44e0-a946-a0be360e1d54"

PATIENT_LABEL = "Patient 10"

# Fixed blinded mapping
blinded_summaries = {
    "A": pilot_summaries["direct"],
    "B": pilot_summaries["hierarchical"],
    "C": pilot_summaries["rag"],
    "D": pilot_summaries["rag_verification"],
}

lines = []

lines.append(PATIENT_LABEL)
lines.append("")
lines.append("SOURCE CLINICAL RECORD")
lines.append(f"Total clinical notes: {len(pilot_notes)}")
lines.append("")

# Exact source notes — unchanged
for note_number, (_, row) in enumerate(pilot_notes.iterrows(), start=1):
    lines.append(f"Clinical Note {note_number}")
    lines.append(f"Date/Time: {row['creation_timestamp']}")
    lines.append("")
    lines.append(str(row["clean_note_text"]).strip())
    lines.append("")
    lines.append("-" * 80)
    lines.append("")

# Exact frozen summaries
lines.append("GENERATED SUMMARIES")
lines.append("")

for label, summary in blinded_summaries.items():
    lines.append(f"SUMMARY {label}")
    lines.append("")
    lines.append(summary.strip())
    lines.append("")
    lines.append("-" * 140)
    lines.append("")

patient_10_content = "\n".join(lines)

print(patient_10_content)

Patient 10

SOURCE CLINICAL RECORD
Total clinical notes: 10

Clinical Note 1
Date/Time: 02/01/2026 18:15

Clerking Doctor
Dr. Risha Mandal (Consultant)

Presenting Complaint
End-stage OA of the left hip.

Procedure
Total left hip replacement.

On Examination
Patient appears well and comfortable at rest, with no signs of distress. Focused examination of the left hip reveals marked restriction in ROM with pain on flexion, internal rotation, and abduction. No erythema, swelling, or skin lesions observed over the surgical site. Neurological examination of the lower limbs demonstrates intact motor and sensory function bilaterally. Reflexes are preserved and dsymmetrical. Assessment of mobility indicates significant limitations in ambulation; patient uses a single walking stick for support. No calf tenderness or swelling noted bilaterally, and no signs suggestive of DVT. Mild hearing loss and a history of lower back pain are noted from the patient's PMH.

Observations
HR 76
BP 136/84
RR 14
T

In [ ]:
| Patient | Source notes | Source words | Direct words | Hierarchical words | RAG words | RAG+V words |
| ------- | -----------: | -----------: | -----------: | -----------------: | --------: | ----------: |
| 01      |           15 |        1,903 |            — |                  — |         — |           — |
| 02      |           23 |        2,958 |          582 |                484 |       477 |         458 |
| 03      |           22 |        3,436 |          418 |                512 |       326 |         326 |
| 04      |           34 |        3,867 |          453 |                496 |       467 |         467 |
| 05      |           36 |        3,991 |          385 |                415 |       351 |         351 |
| 06      |           26 |        4,287 |          584 |                597 |       414 |         408 |
| 07      |           45 |        5,134 |          448 |                468 |       336 |         317 |
| 08      |           35 |        5,525 |          482 |                531 |       490 |         488 |
| 09      |           37 |        4,278 |          404 |                580 |       406 |         394 |
| 10      |           10 |        1,772 |          471 |                415 |       503 |         490 |
